In [ ]:
# ============================================================
# 07b_survival_analysis_temporal_validation
# Refined Cox survival temporal validation pipeline
# ============================================================
#
# Purpose:
# - Perform discharge-time survival prediction for 90-day post-discharge mortality.
# - Use only the 0–7 feature dataset generated by notebook 05b.
# - Preserve temporal validation:
#     Cohort 1 = training
#     Cohort 2 = temporal testing
# - Compare:
#     Cox_A_clinical
#     Cox_B_clinical_labs_trajectory
# - Evaluate:
#     C-index
#     bootstrap confidence intervals
#     hazard ratios
#     proportional hazards assumption
#     Kaplan–Meier risk stratification in Cohort 2
#
# Key methodological point:
# Since survival follow-up starts at hospital discharge, all 0–7 hospitalization
# features are available before the survival time origin. Therefore, the 0–7
# dataset is appropriate for discharge-time risk prediction.
# ============================================================

In [ ]:
# ============================================================
# 01. IMPORTS
# ============================================================

import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

In [ ]:
# ============================================================
# 02. PATHS
# ============================================================

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/"
    "pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

OUTPUT_05B = (
    PROJECT_CONSTANTS_DIR /
    "outputs" /
    "05b_feature_construction_and_selection"
)

OUTPUT_07B = (
    PROJECT_CONSTANTS_DIR /
    "outputs" /
    "07b_survival_analysis_temporal_validation"
)

OUTPUT_07B.mkdir(parents=True, exist_ok=True)

FEATURES_0_7_PATH = OUTPUT_05B / "05b_features_0_7.parquet"

print("FEATURES_0_7_PATH:", FEATURES_0_7_PATH)
print("OUTPUT_07B:", OUTPUT_07B)

In [ ]:
# ============================================================
# 03. LOAD 0–7 DATASET
# ============================================================

df = pd.read_parquet(FEATURES_0_7_PATH)

print("0–7 dataset:", df.shape)

display(df.head())
display(df.dtypes)

In [ ]:
# ============================================================
# 04. DEFINE SURVIVAL OUTCOME VARIABLES
# ============================================================

# Survival analysis is discharge-time prediction.
# Time origin = hospital discharge date.
# Follow-up horizon = 90 days after discharge.

FOLLOWUP_DATE = pd.to_datetime("2026-05-04")
MAX_FOLLOWUP_DAYS = 90

required_cols = ["endDate", "dateOfDeath"]

missing_required = [
    c for c in required_cols
    if c not in df.columns
]

if len(missing_required) > 0:
    raise ValueError(f"Missing required columns: {missing_required}")

df["endDate"] = pd.to_datetime(df["endDate"], errors="coerce")
df["dateOfDeath"] = pd.to_datetime(df["dateOfDeath"], errors="coerce")

# Days from discharge to death
df["days_discharge_to_death"] = (
    df["dateOfDeath"] - df["endDate"]
).dt.days

# Available administrative follow-up after discharge
df["available_followup_days"] = (
    FOLLOWUP_DATE - df["endDate"]
).dt.days

# Exclude patients without valid discharge date
df_surv = df[
    df["endDate"].notna()
    & df["available_followup_days"].notna()
    & (df["available_followup_days"] > 0)
].copy()

# Exclude in-hospital deaths or deaths recorded on/before discharge.
# Survival target is post-discharge mortality.
df_surv = df_surv[
    df_surv["days_discharge_to_death"].isna()
    | (df_surv["days_discharge_to_death"] > 0)
].copy()

# Event = death after discharge within 90 days
df_surv["event_90d"] = np.where(
    df_surv["days_discharge_to_death"].notna()
    & (df_surv["days_discharge_to_death"] <= MAX_FOLLOWUP_DAYS),
    1,
    0,
)

# Survival time:
# - if event: days from discharge to death
# - if no event: censored at min(90, available follow-up)
df_surv["survival_time_90d"] = np.where(
    df_surv["event_90d"] == 1,
    df_surv["days_discharge_to_death"],
    np.minimum(
        MAX_FOLLOWUP_DAYS,
        df_surv["available_followup_days"],
    ),
)

# Remove impossible or zero survival times
df_surv = df_surv[
    df_surv["survival_time_90d"].notna()
    & (df_surv["survival_time_90d"] > 0)
].copy()

df_surv["event_90d"] = df_surv["event_90d"].astype(int)
df_surv["survival_time_90d"] = df_surv["survival_time_90d"].astype(float)

time_col = "survival_time_90d"
event_col = "event_90d"

print("Survival-ready dataset:", df_surv.shape)
print("Time column:", time_col)
print("Event column:", event_col)

print("\nEvents:")
display(df_surv[event_col].value_counts(dropna=False))

print("\nFollow-up summary:")
display(
    df_surv[
        [
            "survival_time_90d",
            "event_90d",
            "available_followup_days",
            "days_discharge_to_death",
        ]
    ].describe()
)

print("\nTemporal cohorts:")
display(df_surv["temporal_cohort"].value_counts(dropna=False))

print("\nMetastatic groups:")
display(df_surv["metastatic_group"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 05. COHORT DEFINITIONS
# ============================================================

COHORT_DEFINITIONS = {
    "full": lambda df: df.index,
    "non_metastatic": lambda df: df.index[df["metastatic_group"] == "Non-metastatic"],
    "metastatic": lambda df: df.index[df["metastatic_group"] == "Metastatic"],
}

for cohort_name, cohort_fn in COHORT_DEFINITIONS.items():
    idx = cohort_fn(df_surv)
    tmp = df_surv.loc[idx].copy()

    print("\n" + "=" * 80)
    print(cohort_name)
    print("=" * 80)
    print("N:", tmp.shape[0])
    print("Events:", int(tmp[event_col].sum()))
    print("Event rate:", tmp[event_col].mean())
    display(tmp["temporal_cohort"].value_counts(dropna=False))

In [ ]:
# ============================================================
# 06. DEFINE FEATURE GROUPS
# ============================================================

clinical_core_features = [
    "age",
    "sex",
    "bmi",
    "adm_mews",
    "admBarthelScore",
    "admBradenScore",
    "admECOG",
    "ccsi",
    "solidTumor_cat",
]

clinical_core_features = [
    c for c in clinical_core_features
    if c in df_surv.columns
]

survival_extra_features = [
    c for c in ["LOS_days"]
    if c in df_surv.columns
]

full_lab_features = [
    c for c in df_surv.columns
    if (
        "_mean_0_7" in c
        or "_delta_0_7" in c
        or "_trajectory_direction_0_7" in c
    )
]

print("Clinical features:", len(clinical_core_features))
print(clinical_core_features)

print("\nSurvival extra features:", survival_extra_features)

print("\n0–7 lab/trajectory features:", len(full_lab_features))
display(full_lab_features)

In [ ]:
# ============================================================
# 07. DEFINE COX FEATURE SETS
# ============================================================

FEATURE_SETS = {
    "Cox_A_clinical": (
        clinical_core_features
        + survival_extra_features
    ),
    "Cox_B_clinical_labs_trajectory": (
        clinical_core_features
        + survival_extra_features
        + full_lab_features
    ),
}

for name, features in FEATURE_SETS.items():
    features = list(dict.fromkeys(features))
    FEATURE_SETS[name] = features

    print(name, len(features))
    display(features)

In [ ]:
# ============================================================
# 08. PREPROCESSING HELPERS
# ============================================================

def clean_X_for_model(X):
    X = X.copy()
    X = X.loc[:, ~X.columns.duplicated()].copy()

    for col in X.columns:
        if X[col].dtype == "bool":
            X[col] = X[col].astype(int)

        elif (
            X[col].dtype == "object"
            or str(X[col].dtype) in ["string", "category"]
        ):
            X[col] = (
                X[col]
                .astype("string")
                .fillna("Missing")
                .astype(str)
            )

        else:
            X[col] = pd.to_numeric(X[col], errors="coerce")

    return X


def build_preprocessor(X):
    X = clean_X_for_model(X)

    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

    categorical_features = [
        c for c in X.columns
        if c not in numeric_features
    ]

    numeric_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", RobustScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
    )

    return preprocessor


def get_feature_names(preprocessor):
    feature_names = []

    num_features = preprocessor.transformers_[0][2]
    cat_features = preprocessor.transformers_[1][2]

    num_pipeline = preprocessor.named_transformers_["num"]
    cat_pipeline = preprocessor.named_transformers_["cat"]

    num_imputer = num_pipeline.named_steps["imputer"]

    numeric_output_names = list(num_features)

    if hasattr(num_imputer, "indicator_"):
        indicator_names = [
            f"{num_features[i]}_missing_indicator"
            for i in num_imputer.indicator_.features_
        ]

        numeric_output_names += indicator_names

    if len(cat_features) > 0:
        cat_encoder = cat_pipeline.named_steps["onehot"]
        cat_output_names = list(
            cat_encoder.get_feature_names_out(cat_features)
        )
    else:
        cat_output_names = []

    feature_names = numeric_output_names + cat_output_names

    return feature_names


def preprocess_train_test(X_train, X_test):
    X_train = clean_X_for_model(X_train)
    X_test = clean_X_for_model(X_test)

    X_test = X_test[X_train.columns]

    preprocessor = build_preprocessor(X_train)

    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    feature_names = get_feature_names(preprocessor)

    X_train_processed = pd.DataFrame(
        X_train_processed,
        columns=feature_names,
        index=X_train.index,
    )

    X_test_processed = pd.DataFrame(
        X_test_processed,
        columns=feature_names,
        index=X_test.index,
    )

    return X_train_processed, X_test_processed, preprocessor

In [ ]:
# ============================================================
# 09. COX HELPER FUNCTIONS
# ============================================================

def fit_cox_model(
    train_df,
    duration_col,
    event_col,
    penalizer=0.1,
):
    cph = CoxPHFitter(
        penalizer=penalizer
    )

    cph.fit(
        train_df,
        duration_col=duration_col,
        event_col=event_col,
        show_progress=False,
    )

    return cph


def compute_cindex(cph, df_eval, duration_col, event_col):
    risk_scores = cph.predict_partial_hazard(df_eval)

    cindex = concordance_index(
        event_times=df_eval[duration_col],
        predicted_scores=-risk_scores,
        event_observed=df_eval[event_col],
    )

    return cindex


def bootstrap_cindex_ci(
    cph,
    df_eval,
    duration_col,
    event_col,
    n_bootstrap=500,
    seed=42,
):
    rng = np.random.default_rng(seed)

    values = []
    n = df_eval.shape[0]

    for _ in range(n_bootstrap):
        idx = rng.choice(
            np.arange(n),
            size=n,
            replace=True,
        )

        sample = df_eval.iloc[idx].copy()

        if sample[event_col].nunique() < 2:
            continue

        try:
            values.append(
                compute_cindex(
                    cph=cph,
                    df_eval=sample,
                    duration_col=duration_col,
                    event_col=event_col,
                )
            )

        except Exception:
            continue

    if len(values) == 0:
        return np.nan, np.nan

    return (
        np.percentile(values, 2.5),
        np.percentile(values, 97.5),
    )


def prepare_cox_dataframe(
    X_processed,
    durations,
    events,
    duration_col,
    event_col,
):
    out = X_processed.copy()
    out[duration_col] = durations
    out[event_col] = events

    return out

In [ ]:
# ============================================================
# 10. MAIN TEMPORAL VALIDATION LOOP
# ============================================================

cox_results = []
cox_predictions = []
cox_models = {}

PENALIZER = 0.1

for cohort_name, cohort_fn in COHORT_DEFINITIONS.items():

    cohort_idx = cohort_fn(df_surv)
    df_sub = df_surv.loc[cohort_idx].copy()

    train_df = df_sub[
        df_sub["temporal_cohort"] == "Cohort 1"
    ].copy()

    test_df = df_sub[
        df_sub["temporal_cohort"] == "Cohort 2"
    ].copy()

    if train_df.empty or test_df.empty:
        continue

    if train_df[event_col].nunique() < 2 or test_df[event_col].nunique() < 2:
        continue

    print("\n" + "=" * 100)
    print("COHORT:", cohort_name)
    print("=" * 100)
    print("Train:", train_df.shape[0], "events:", int(train_df[event_col].sum()))
    print("Test :", test_df.shape[0], "events:", int(test_df[event_col].sum()))

    for feature_set_name, features in FEATURE_SETS.items():

        features = [
            f for f in features
            if f in df_sub.columns
        ]

        print("\nFeature set:", feature_set_name)
        print("N features:", len(features))

        X_train = train_df[features].copy()
        X_test = test_df[features].copy()

        X_train_p, X_test_p, preprocessor = preprocess_train_test(
            X_train,
            X_test,
        )

        cox_train = prepare_cox_dataframe(
            X_processed=X_train_p,
            durations=train_df[time_col].values,
            events=train_df[event_col].values,
            duration_col=time_col,
            event_col=event_col,
        )

        cox_test = prepare_cox_dataframe(
            X_processed=X_test_p,
            durations=test_df[time_col].values,
            events=test_df[event_col].values,
            duration_col=time_col,
            event_col=event_col,
        )

        try:
            cph = fit_cox_model(
                train_df=cox_train,
                duration_col=time_col,
                event_col=event_col,
                penalizer=PENALIZER,
            )

        except Exception as e:
            print("FAILED:", cohort_name, feature_set_name, e)
            continue

        train_cindex = compute_cindex(
            cph=cph,
            df_eval=cox_train,
            duration_col=time_col,
            event_col=event_col,
        )

        test_cindex = compute_cindex(
            cph=cph,
            df_eval=cox_test,
            duration_col=time_col,
            event_col=event_col,
        )

        test_ci_low, test_ci_high = bootstrap_cindex_ci(
            cph=cph,
            df_eval=cox_test,
            duration_col=time_col,
            event_col=event_col,
            n_bootstrap=500,
        )

        train_risk = cph.predict_partial_hazard(cox_train).values.flatten()
        test_risk = cph.predict_partial_hazard(cox_test).values.flatten()

        row = {
            "cohort": cohort_name,
            "feature_set": feature_set_name,
            "n_features_original": len(features),
            "n_features_processed": X_train_p.shape[1],
            "penalizer": PENALIZER,
            "train_n": train_df.shape[0],
            "train_events": int(train_df[event_col].sum()),
            "train_event_rate": train_df[event_col].mean(),
            "test_n": test_df.shape[0],
            "test_events": int(test_df[event_col].sum()),
            "test_event_rate": test_df[event_col].mean(),
            "train_cindex": train_cindex,
            "test_cindex": test_cindex,
            "test_cindex_ci_lower": test_ci_low,
            "test_cindex_ci_upper": test_ci_high,
        }

        cox_results.append(row)

        pred_train = pd.DataFrame({
            "episode_key": train_df["episode_key"].values,
            "pubID": train_df["pubID"].values,
            "cohort": cohort_name,
            "feature_set": feature_set_name,
            "split": "train",
            "duration": train_df[time_col].values,
            "event": train_df[event_col].values,
            "risk_score": train_risk,
            "temporal_cohort": train_df["temporal_cohort"].values,
            "metastatic_group": train_df["metastatic_group"].values,
        })

        pred_test = pd.DataFrame({
            "episode_key": test_df["episode_key"].values,
            "pubID": test_df["pubID"].values,
            "cohort": cohort_name,
            "feature_set": feature_set_name,
            "split": "test",
            "duration": test_df[time_col].values,
            "event": test_df[event_col].values,
            "risk_score": test_risk,
            "temporal_cohort": test_df["temporal_cohort"].values,
            "metastatic_group": test_df["metastatic_group"].values,
        })

        cox_predictions.append(pred_train)
        cox_predictions.append(pred_test)

        cox_models[(cohort_name, feature_set_name)] = {
            "model": cph,
            "features": features,
            "preprocessor": preprocessor,
            "X_train_processed": X_train_p,
            "X_test_processed": X_test_p,
            "cox_train": cox_train,
            "cox_test": cox_test,
            "train_df": train_df,
            "test_df": test_df,
            "train_risk": train_risk,
            "test_risk": test_risk,
        }

cox_results_df = pd.DataFrame(cox_results)
cox_predictions_df = pd.concat(cox_predictions, ignore_index=True)

display(cox_results_df)
display(cox_predictions_df.head())

In [ ]:
# ============================================================
# 11. BEST MODEL SELECTION
# ============================================================

best_cox_models = (
    cox_results_df
    .sort_values("test_cindex", ascending=False)
    .groupby("cohort", as_index=False)
    .first()
)

display(best_cox_models)

In [ ]:
# ============================================================
# 12. CLINICAL VS LAB/TRAJECTORY COMPARISON
# ============================================================

comparison_rows = []

for cohort_name, tmp in cox_results_df.groupby("cohort"):

    clinical = tmp[
        tmp["feature_set"] == "Cox_A_clinical"
    ]

    labs = tmp[
        tmp["feature_set"] == "Cox_B_clinical_labs_trajectory"
    ]

    if clinical.empty or labs.empty:
        continue

    clinical = clinical.iloc[0]
    labs = labs.iloc[0]

    comparison_rows.append({
        "cohort": cohort_name,
        "clinical_cindex": clinical["test_cindex"],
        "labs_cindex": labs["test_cindex"],
        "delta_cindex_labs_minus_clinical": (
            labs["test_cindex"] - clinical["test_cindex"]
        ),
        "clinical_ci_lower": clinical["test_cindex_ci_lower"],
        "clinical_ci_upper": clinical["test_cindex_ci_upper"],
        "labs_ci_lower": labs["test_cindex_ci_lower"],
        "labs_ci_upper": labs["test_cindex_ci_upper"],
    })

cox_comparison_df = pd.DataFrame(comparison_rows)

display(cox_comparison_df)

In [ ]:
# ============================================================
# 13. PLOT C-INDEX COMPARISON
# ============================================================

plot_df = cox_results_df.copy()

plot_df["label"] = (
    plot_df["cohort"] + " | " + plot_df["feature_set"]
)

plot_df = plot_df.sort_values("test_cindex")

x = plot_df["test_cindex"]
xerr_low = x - plot_df["test_cindex_ci_lower"]
xerr_high = plot_df["test_cindex_ci_upper"] - x

plt.figure(figsize=(9, 6))

plt.errorbar(
    x=x,
    y=np.arange(len(plot_df)),
    xerr=[xerr_low, xerr_high],
    fmt="o",
    capsize=4,
)

plt.yticks(
    np.arange(len(plot_df)),
    plot_df["label"],
)

plt.axvline(0.5, linestyle="--")
plt.axvline(0.7, linestyle="--")

plt.xlabel("Temporal test C-index with 95% bootstrap CI")
plt.title("Cox survival temporal validation")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 14. HAZARD RATIOS FOR BEST MODELS
# ============================================================

hazard_ratio_rows = []

for _, r in best_cox_models.iterrows():

    cohort_name = r["cohort"]
    feature_set_name = r["feature_set"]

    obj = cox_models[(cohort_name, feature_set_name)]
    cph = obj["model"]

    summary = cph.summary.reset_index().rename(
        columns={"covariate": "feature"}
    )

    summary["cohort"] = cohort_name
    summary["feature_set"] = feature_set_name

    hazard_ratio_rows.append(summary)

hazard_ratios_df = pd.concat(
    hazard_ratio_rows,
    ignore_index=True,
)

display(
    hazard_ratios_df[
        [
            "cohort",
            "feature_set",
            "feature",
            "coef",
            "exp(coef)",
            "exp(coef) lower 95%",
            "exp(coef) upper 95%",
            "p",
        ]
    ]
    .sort_values("p")
    .head(50)
)

In [ ]:
# ============================================================
# 15. TRAIN-BASED RISK GROUP ASSIGNMENT
# ============================================================

def assign_train_based_tertiles(train_risk, eval_risk):
    q1, q2 = np.quantile(
        train_risk,
        [1 / 3, 2 / 3],
    )

    def assign(x):
        if x <= q1:
            return "Low"
        elif x <= q2:
            return "Intermediate"
        else:
            return "High"

    return pd.Series(eval_risk).apply(assign).values


risk_group_rows = []

for _, r in best_cox_models.iterrows():

    cohort_name = r["cohort"]
    feature_set_name = r["feature_set"]

    obj = cox_models[(cohort_name, feature_set_name)]

    train_groups = assign_train_based_tertiles(
        train_risk=obj["train_risk"],
        eval_risk=obj["train_risk"],
    )

    test_groups = assign_train_based_tertiles(
        train_risk=obj["train_risk"],
        eval_risk=obj["test_risk"],
    )

    train_pred = cox_predictions_df[
        (cox_predictions_df["cohort"] == cohort_name)
        & (cox_predictions_df["feature_set"] == feature_set_name)
        & (cox_predictions_df["split"] == "train")
    ].copy()

    test_pred = cox_predictions_df[
        (cox_predictions_df["cohort"] == cohort_name)
        & (cox_predictions_df["feature_set"] == feature_set_name)
        & (cox_predictions_df["split"] == "test")
    ].copy()

    train_pred["risk_group"] = train_groups
    test_pred["risk_group"] = test_groups

    risk_group_rows.append(train_pred)
    risk_group_rows.append(test_pred)

risk_predictions_df = pd.concat(
    risk_group_rows,
    ignore_index=True,
)

display(risk_predictions_df.head())

In [ ]:
# ============================================================
# 16. RISK GROUP SUMMARY IN COHORT 2
# ============================================================

risk_summary_df = (
    risk_predictions_df[
        risk_predictions_df["split"] == "test"
    ]
    .groupby(
        ["cohort", "feature_set", "risk_group"],
        observed=True,
    )
    .agg(
        n=("event", "size"),
        events=("event", "sum"),
        event_rate=("event", "mean"),
        median_risk_score=("risk_score", "median"),
        mean_risk_score=("risk_score", "mean"),
    )
    .reset_index()
)

risk_summary_df["risk_group"] = pd.Categorical(
    risk_summary_df["risk_group"],
    categories=["Low", "Intermediate", "High"],
    ordered=True,
)

risk_summary_df = risk_summary_df.sort_values(
    ["cohort", "feature_set", "risk_group"]
)

display(risk_summary_df)

In [ ]:
# ============================================================
# 17. KAPLAN–MEIER CURVES IN COHORT 2
# ============================================================

kmf = KaplanMeierFitter()

for (cohort_name, feature_set_name), group_df in risk_predictions_df[
    risk_predictions_df["split"] == "test"
].groupby(["cohort", "feature_set"]):

    plt.figure(figsize=(7, 5))

    for risk_group in ["Low", "Intermediate", "High"]:

        tmp = group_df[
            group_df["risk_group"] == risk_group
        ].copy()

        if tmp.empty:
            continue

        kmf.fit(
            durations=tmp["duration"],
            event_observed=tmp["event"],
            label=f"{risk_group} risk",
        )

        kmf.plot_survival_function(ci_show=True)

    plt.title(
        f"Kaplan–Meier curves in Cohort 2\n"
        f"{cohort_name} | {feature_set_name}"
    )

    plt.xlabel("Days after discharge")
    plt.ylabel("Survival probability")
    plt.ylim(0, 1.02)
    plt.tight_layout()

    output_path = (
        OUTPUT_07B /
        f"KM_test_{cohort_name}_{feature_set_name}.png"
    )

    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# 18. LOG-RANK TESTING
# ============================================================

logrank_rows = []

for (cohort_name, feature_set_name), group_df in risk_predictions_df[
    risk_predictions_df["split"] == "test"
].groupby(["cohort", "feature_set"]):

    try:
        global_test = multivariate_logrank_test(
            event_durations=group_df["duration"],
            groups=group_df["risk_group"],
            event_observed=group_df["event"],
        )

        logrank_rows.append({
            "cohort": cohort_name,
            "feature_set": feature_set_name,
            "comparison": "global",
            "test_statistic": global_test.test_statistic,
            "p_value": global_test.p_value,
        })

    except Exception:
        pass

    pairs = [
        ("Low", "Intermediate"),
        ("Low", "High"),
        ("Intermediate", "High"),
    ]

    for g1, g2 in pairs:

        d1 = group_df[group_df["risk_group"] == g1]
        d2 = group_df[group_df["risk_group"] == g2]

        if d1.empty or d2.empty:
            continue

        try:
            result = logrank_test(
                durations_A=d1["duration"],
                durations_B=d2["duration"],
                event_observed_A=d1["event"],
                event_observed_B=d2["event"],
            )

            logrank_rows.append({
                "cohort": cohort_name,
                "feature_set": feature_set_name,
                "comparison": f"{g1} vs {g2}",
                "test_statistic": result.test_statistic,
                "p_value": result.p_value,
            })

        except Exception:
            continue

logrank_df = pd.DataFrame(logrank_rows)

display(logrank_df)

In [ ]:
# ============================================================
# 19. PROPORTIONAL HAZARDS DIAGNOSTICS
# ============================================================

for _, r in best_cox_models.iterrows():

    cohort_name = r["cohort"]
    feature_set_name = r["feature_set"]

    obj = cox_models[(cohort_name, feature_set_name)]

    print("\n" + "=" * 100)
    print("PH diagnostics:", cohort_name, "|", feature_set_name)
    print("=" * 100)

    try:
        obj["model"].check_assumptions(
            obj["cox_train"],
            p_value_threshold=0.05,
            show_plots=False,
        )

    except Exception as e:
        print("PH diagnostic failed:", e)

In [ ]:
# ============================================================
# 20. INTERPRETATION FLAGS
# ============================================================

interpretation_df = cox_results_df.copy()

interpretation_df["cindex_ci_width"] = (
    interpretation_df["test_cindex_ci_upper"]
    - interpretation_df["test_cindex_ci_lower"]
)

interpretation_df["cindex_interpretation"] = pd.cut(
    interpretation_df["test_cindex"],
    bins=[0, 0.6, 0.7, 0.8, 1.0],
    labels=[
        "Weak",
        "Moderate",
        "Good",
        "Very good",
    ],
    include_lowest=True,
)

interpretation_df["uncertainty_interpretation"] = pd.cut(
    interpretation_df["cindex_ci_width"],
    bins=[0, 0.15, 0.25, 1.0],
    labels=[
        "Relatively stable",
        "Moderate uncertainty",
        "High uncertainty",
    ],
    include_lowest=True,
)

display(
    interpretation_df[
        [
            "cohort",
            "feature_set",
            "test_n",
            "test_events",
            "test_event_rate",
            "test_cindex",
            "test_cindex_ci_lower",
            "test_cindex_ci_upper",
            "cindex_interpretation",
            "cindex_ci_width",
            "uncertainty_interpretation",
        ]
    ].sort_values(["cohort", "feature_set"])
)

In [ ]:
# ============================================================
# 21. SAVE OUTPUTS
# ============================================================

results_path = OUTPUT_07B / "07b_cox_temporal_validation_results.xlsx"
predictions_path = OUTPUT_07B / "07b_cox_predictions_train_test.parquet"
risk_predictions_path = OUTPUT_07B / "07b_cox_risk_group_predictions.parquet"

cox_predictions_df.to_parquet(
    predictions_path,
    index=False,
)

risk_predictions_df.to_parquet(
    risk_predictions_path,
    index=False,
)

with pd.ExcelWriter(results_path) as writer:

    cox_results_df.to_excel(
        writer,
        sheet_name="all_models",
        index=False,
    )

    best_cox_models.to_excel(
        writer,
        sheet_name="best_models",
        index=False,
    )

    cox_comparison_df.to_excel(
        writer,
        sheet_name="clinical_vs_labs",
        index=False,
    )

    risk_summary_df.to_excel(
        writer,
        sheet_name="risk_groups",
        index=False,
    )

    logrank_df.to_excel(
        writer,
        sheet_name="logrank_tests",
        index=False,
    )

    hazard_ratios_df.to_excel(
        writer,
        sheet_name="hazard_ratios",
        index=False,
    )

    interpretation_df.to_excel(
        writer,
        sheet_name="interpretation_summary",
        index=False,
    )

print("Saved:")
print(results_path)
print(predictions_path)
print(risk_predictions_path)

In [ ]:
# ============================================================
# 22. FINAL NOTE
# ============================================================

print("""
Notebook 07b completed.

This refined survival modelling notebook performs discharge-time 90-day survival
prediction using the 0–7 feature dataset generated by notebook 05b.

Main methodological choices:
- survival time origin is hospital discharge;
- all 0–7 features are available before the survival prediction point;
- temporal validation is preserved using Cohort 1 for training and Cohort 2 for testing;
- Cox_A_clinical is compared with Cox_B_clinical_labs_trajectory;
- risk groups are assigned using thresholds learned only in the training cohort;
- Kaplan–Meier curves and log-rank tests are evaluated in Cohort 2.

The notebook prioritizes clinical interpretability, temporal robustness, and
methodological consistency with the refined supervised modelling pipeline.
""")